# Step 2 - Prepare: labels, split, per-segment direction, merged rows, shortcut check

Reads the Parquet parts from step 1 (the zip is not needed any more) and, per file:
finds the device IP of every capture segment (some "-1" files are several captures joined end to end) and recomputes `dir_in` / `dir_out`;
maps raw labels to the six classes and drops `Unknown`; tags every packet with its split and its episode (the n-th run of its attack);
merges packets that are identical in every feature, label, split and episode, keeping the number of copies as `weight`.
Then it writes `reports/prepare/PREPARE.md` with rows per class and split, a **shortcut check** (how well each feature alone separates the classes on validation) and the segment table.

* **Runtime:** CPU (*Runtime -> Change runtime type -> CPU*).
* **Time:** roughly 15-30 min.
* **Disconnected?** Reconnect and run cells 1, 2 and 4 again; finished files are skipped.
* **Results:** copy what cell 6 prints into the chat.


In [ ]:
#@title 1. Settings
REPO_URL = "https://github.com/madhumithakk01/ds.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
ZIP_PATH = "/content/drive/MyDrive/Ds/GothamDataset2025.zip"  # "Ds" = shortcut to the shared folder  #@param {type:"string"}
WORK_DIR = "/content/drive/MyDrive/gotham_ids"  #@param {type:"string"}

In [ ]:
#@title 2. Mount Drive, get the latest code, check the environment
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = "/content/ds"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")], check=True)

os.chdir(REPO_DIR)
if not os.path.exists(ZIP_PATH):
    folder = os.path.dirname(ZIP_PATH)
    print(f"ZIP_PATH not found: {ZIP_PATH}")
    if os.path.isdir(folder):
        print(f"Contents of {folder}:")
        for name in sorted(os.listdir(folder))[:50]:
            print("  ", name)
    print("Set ZIP_PATH in cell 1 to the zip shown above (the code will also search My Drive for it).")
os.environ["GIDS_WORK_DIR"] = WORK_DIR
os.environ["GIDS_ZIP_PATH"] = ZIP_PATH
print(subprocess.run(["git", "log", "-1", "--format=code version %h (%cd)"], capture_output=True, text=True).stdout)
!python -m gids env

In [ ]:
#@title 3. Self-test on a small synthetic archive (about 1 min)
!python -m gids selftest


In [ ]:
#@title 4. Prepare every ingested file, then write the report (resumable: after a disconnect run cells 1, 2 and 4 again)
!python -u -m gids prepare


In [ ]:
#@title 5. Report
from IPython.display import Markdown, display
display(Markdown(open(f"{WORK_DIR}/reports/prepare/PREPARE.md").read()))


In [ ]:
#@title 6. Plain text of the report (copy all of it into the chat)
print(open(f"{WORK_DIR}/reports/prepare/PREPARE.md").read())


### If something goes wrong
* `status` shows how many files are prepared.
* `trace --errors` prints warnings and errors of the latest run with full tracebacks.
* Changing `labels`, `split` or `prepare.drop_features` writes to a new `prepare-<hash>` folder; the old one stays for comparison.


In [ ]:
!python -m gids status
!python -m gids trace --errors --tail 30
